# 09｜Choice财务报表与分红小样本验收

本Notebook只抓取3只证券、2个报告期，用于确认Choice财务与分红权限、指标口径、落库结构和重复运行幂等性，不进行全市场下载。

默认样本：

- `000001.SZ`：金融行业样本；
- `600519.SH`：沪市非金融样本；
- `300750.SZ`：深市非金融样本；
- 报告期：`2025-12-31`、`2026-06-30`。

Choice官方说明：`css`用于截面财务等数据；`ReportDate`是季度最后一个自然日，并非实际公告披露日。不同账号、指标版本和购买权限可能返回不同结果，因此本Notebook会先逐个探测候选指标，再下载通过探测的指标。

官方资料：

- https://quantapi.eastmoney.com/Upload/EMQuantAPI_Python.html
- https://quantapi.eastmoney.com/Cmd/ChoiceSerialSection?from=web

重要：金额和比例暂时保存为`unit=vendor_raw`。在Choice命令生成器确认单位前，不擅自认定为“元”“万元”或百分比小数。


In [ ]:
from pathlib import Path
import os
import sys


def locate_project_root():
    configured = os.getenv("QIANJI_PROJECT_ROOT", "").strip()
    candidates = [Path(configured)] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, current.parent, *current.parents])
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "qianji_data_mini").exists():
            return candidate.resolve()
    raise RuntimeError("未找到项目根目录。请设置QIANJI_PROJECT_ROOT。")


PROJECT_ROOT = locate_project_root()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Python路径：", sys.executable)
print("项目根目录：", PROJECT_ROOT)
print("Notebook当前目录：", Path.cwd().resolve())


In [ ]:
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

try:
    installed_version = version("qianji-data-mini")
except PackageNotFoundError:
    installed_version = "0.0.0"

print("qianji-data-mini版本：", installed_version)
if Version(installed_version) < Version("0.7.0"):
    raise RuntimeError(
        "当前版本低于0.7.0。请覆盖09号配套补丁，运行00号Notebook，重启内核后再运行。"
    )

try:
    from EmQuantAPI import c
    print("EmQuantAPI：导入成功")
except Exception as exc:
    raise RuntimeError(f"EmQuantAPI导入失败：{type(exc).__name__}: {exc}") from exc


## 参数与候选指标

候选指标会逐个调用Choice进行探测，只有成功返回且结构可解析的指标才进入正式小样本下载。被拒指标不等同于项目失败；它们用于形成账号权限和指标版本证据。

如果某一数据集没有任何候选指标通过，请在Choice“命令生成”中生成该指标的`css`命令，把英文指标简称填入对应`.env`变量后重跑。


In [ ]:
from datetime import date, datetime, timezone

from qianji_data_mini.financial_ingest import DEFAULT_INDICATOR_CANDIDATES


def csv_items(name, defaults):
    raw = os.getenv(name, ",".join(defaults))
    return list(dict.fromkeys(item.strip().upper() for item in raw.split(",") if item.strip()))


SYMBOLS = csv_items("CHOICE_FINANCIAL_SAMPLE_SYMBOLS", ["000001.SZ", "600519.SH", "300750.SZ"])
REPORT_DATES = csv_items("CHOICE_FINANCIAL_REPORT_DATES", ["2025-12-31", "2026-06-30"])
INDICATOR_CANDIDATES = {
    "income": csv_items("CHOICE_INCOME_INDICATORS", DEFAULT_INDICATOR_CANDIDATES["income"]),
    "balance": csv_items("CHOICE_BALANCE_INDICATORS", DEFAULT_INDICATOR_CANDIDATES["balance"]),
    "cashflow": csv_items("CHOICE_CASHFLOW_INDICATORS", DEFAULT_INDICATOR_CANDIDATES["cashflow"]),
    "dividend": csv_items("CHOICE_DIVIDEND_INDICATORS", DEFAULT_INDICATOR_CANDIDATES["dividend"]),
}
FINANCIAL_OPTIONS_TEMPLATE = os.getenv(
    "CHOICE_FINANCIAL_OPTIONS_TEMPLATE", "ReportDate={report_date},type=1"
)
DIVIDEND_OPTIONS_TEMPLATE = os.getenv(
    "CHOICE_DIVIDEND_OPTIONS_TEMPLATE", "ReportDate={report_date},PayYear={year}"
)
PROBE_INDICATORS = os.getenv("CHOICE_FINANCIAL_PROBE_INDICATORS", "1") == "1"
RUN_IDEMPOTENCY_CHECK = os.getenv("CHOICE_FINANCIAL_RUN_TWICE", "1") == "1"
CREATE_BACKUP = os.getenv("CHOICE_FINANCIAL_CREATE_BACKUP", "1") == "1"
STRICT_MODE = os.getenv("CHOICE_FINANCIAL_STRICT", "0") == "1"
ALLOW_LARGE_SAMPLE = os.getenv("CHOICE_FINANCIAL_ALLOW_LARGE_SAMPLE", "0") == "1"

configured_db = Path(os.getenv("QIANJI_DB_PATH", "data/qianji_market.db"))
DATABASE_PATH = (
    configured_db.resolve()
    if configured_db.is_absolute()
    else (PROJECT_ROOT / configured_db).resolve()
)
OUTPUT_DIR = PROJECT_ROOT / "validation_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not ALLOW_LARGE_SAMPLE and (len(SYMBOLS) > 5 or len(REPORT_DATES) > 8):
    raise ValueError("09号默认只允许最多5只证券、8个报告期；不要直接改成全市场下载。")
for value in REPORT_DATES:
    date.fromisoformat(value)

safe_config = {
    "symbols": SYMBOLS,
    "report_dates": REPORT_DATES,
    "indicator_candidates": INDICATOR_CANDIDATES,
    "financial_options_template": FINANCIAL_OPTIONS_TEMPLATE,
    "dividend_options_template": DIVIDEND_OPTIONS_TEMPLATE,
    "probe_indicators": PROBE_INDICATORS,
    "run_idempotency_check": RUN_IDEMPOTENCY_CHECK,
    "database_path": str(DATABASE_PATH),
    "choice_login_mode": os.getenv("CHOICE_LOGIN_MODE", "auto"),
    "choice_username_configured": bool(os.getenv("CHOICE_USERNAME")),
    "choice_password_configured": bool(os.getenv("CHOICE_PASSWORD")),
}
safe_config


## 一致性备份与刷新前计数

In [ ]:
import sqlite3

from qianji_data_mini import Database

database = Database(DATABASE_PATH)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = None
if CREATE_BACKUP and DATABASE_PATH.exists():
    backup_dir = OUTPUT_DIR / "backups"
    backup_dir.mkdir(parents=True, exist_ok=True)
    backup_path = backup_dir / f"qianji_market_before_choice_financial_{run_timestamp}.db"
    with database.connect() as source_connection, sqlite3.connect(backup_path) as target_connection:
        source_connection.backup(target_connection)
    print("数据库一致性备份：", backup_path)


def scalar(sql, params=()):
    with database.connect() as connection:
        return connection.execute(sql, params).fetchone()[0]


def scoped_counts(stage):
    symbol_placeholders = ",".join("?" for _ in SYMBOLS)
    date_placeholders = ",".join("?" for _ in REPORT_DATES)
    params = [*SYMBOLS, *REPORT_DATES]
    return {
        "stage": stage,
        "statement_scope": scalar(
            f"SELECT COUNT(*) FROM financial_statement_fact WHERE source='choice' "
            f"AND symbol IN ({symbol_placeholders}) AND report_date IN ({date_placeholders})",
            params,
        ),
        "dividend_scope": scalar(
            f"SELECT COUNT(*) FROM dividend_fact WHERE source='choice' "
            f"AND symbol IN ({symbol_placeholders}) AND report_date IN ({date_placeholders})",
            params,
        ),
        "financial_run_log": scalar("SELECT COUNT(*) FROM financial_ingestion_run"),
    }


counts_before = scoped_counts("刷新前")
counts_before


## 第一次真实探测、下载和落库

本单元会登录Choice。它不会输出账号、密码、token或userinfo内容。


In [ ]:
from qianji_data_mini import ingest_choice_financial_sample

ingest_kwargs = {
    "symbols": SYMBOLS,
    "report_dates": REPORT_DATES,
    "indicator_candidates": INDICATOR_CANDIDATES,
    "financial_options_template": FINANCIAL_OPTIONS_TEMPLATE,
    "dividend_options_template": DIVIDEND_OPTIONS_TEMPLATE,
    "probe_indicators": PROBE_INDICATORS,
    "database_path": DATABASE_PATH,
}

first_result = ingest_choice_financial_sample(**ingest_kwargs)
counts_after_first = scoped_counts("第一次刷新后")
first_summary = first_result.model_dump(mode="json")
first_summary


## 第二次运行：验证幂等

即使第一次有指标被拒，也会执行第二次；只要选中的指标和成功落库部分不重复增长，就能保留幂等证据。


In [ ]:
second_result = None
if RUN_IDEMPOTENCY_CHECK:
    second_result = ingest_choice_financial_sample(**ingest_kwargs)
    print("第二次运行完成。")
else:
    print("已关闭第二次运行。")

counts_after_second = scoped_counts("第二次刷新后" if second_result else "未执行第二次刷新")
second_summary = second_result.model_dump(mode="json") if second_result else None
second_summary


## 读取SQLite证据

In [ ]:
import json
import math
import pandas as pd

database = Database(DATABASE_PATH)
statement_facts = database.query_financial_statement_facts(
    source="choice",
    symbols=SYMBOLS,
    start_report_date=min(REPORT_DATES),
    end_report_date=max(REPORT_DATES),
)
statement_facts = statement_facts[statement_facts["report_date"].isin(REPORT_DATES)].copy()
dividend_facts = database.query_dividend_facts(
    source="choice",
    symbols=SYMBOLS,
    start_report_date=min(REPORT_DATES),
    end_report_date=max(REPORT_DATES),
)
dividend_facts = dividend_facts[dividend_facts["report_date"].isin(REPORT_DATES)].copy()
ingestion_runs = database.query_financial_ingestion_runs().tail(10).copy()
idempotency = pd.DataFrame([counts_before, counts_after_first, counts_after_second])

selected_rows = []
for dataset, indicators in first_result.selected_indicators.items():
    for indicator in indicators:
        selected_rows.append({"dataset": dataset, "indicator": indicator, "status": "selected"})
selected_indicators = pd.DataFrame(selected_rows, columns=["dataset", "indicator", "status"])

rejected_rows = []
for dataset, indicators in first_result.rejected_indicators.items():
    for indicator, reason in indicators.items():
        rejected_rows.append({"dataset": dataset, "indicator": indicator, "status": "rejected", "reason": reason})
rejected_indicators = pd.DataFrame(
    rejected_rows, columns=["dataset", "indicator", "status", "reason"]
)

error_rows = [{"request": key, "error": value} for key, value in first_result.errors.items()]
errors_frame = pd.DataFrame(error_rows, columns=["request", "error"])

print("财务事实：", len(statement_facts))
print("分红事实：", len(dividend_facts))
display(idempotency)
display(selected_indicators)
display(rejected_indicators)
display(errors_frame)


## 完整性与空值分析

空值不自动填0。分红为空可能表示该报告期无方案，也可能表示指标口径或权限不匹配，需要结合“被拒指标”和Choice终端页面判断。


In [ ]:
statement_facts["has_value"] = statement_facts["value_numeric"].notna() | statement_facts["value_text"].notna()
dividend_facts["has_value"] = dividend_facts["value_numeric"].notna() | dividend_facts["value_text"].notna()

statement_completeness = (
    statement_facts.groupby(["statement_type", "indicator"], dropna=False)
    .agg(rows=("symbol", "size"), symbols=("symbol", "nunique"), report_dates=("report_date", "nunique"), non_null=("has_value", "sum"))
    .reset_index()
) if not statement_facts.empty else pd.DataFrame(columns=["statement_type", "indicator", "rows", "symbols", "report_dates", "non_null"])

dividend_completeness = (
    dividend_facts.groupby("indicator", dropna=False)
    .agg(rows=("symbol", "size"), symbols=("symbol", "nunique"), report_dates=("report_date", "nunique"), non_null=("has_value", "sum"))
    .reset_index()
) if not dividend_facts.empty else pd.DataFrame(columns=["indicator", "rows", "symbols", "report_dates", "non_null"])

display(statement_completeness)
display(dividend_completeness)


## 质量门槛

In [ ]:
selected = first_result.selected_indicators
expected_statement_rows = len(SYMBOLS) * len(REPORT_DATES) * sum(len(selected[key]) for key in ["income", "balance", "cashflow"])
expected_dividend_rows = len(SYMBOLS) * len(REPORT_DATES) * len(selected["dividend"])
statement_duplicates = int(statement_facts.duplicated(["source", "symbol", "statement_type", "report_date", "indicator"]).sum()) if not statement_facts.empty else 0
dividend_duplicates = int(dividend_facts.duplicated(["source", "symbol", "report_date", "indicator"]).sum()) if not dividend_facts.empty else 0

numeric_values = pd.concat(
    [statement_facts.get("value_numeric", pd.Series(dtype=float)), dividend_facts.get("value_numeric", pd.Series(dtype=float))],
    ignore_index=True,
).dropna()
finite_numeric = all(math.isfinite(float(value)) for value in numeric_values)
second_core_unchanged = second_result is not None and all(
    counts_after_first[key] == counts_after_second[key]
    for key in ["statement_scope", "dividend_scope"]
)
second_selection_same = second_result is not None and first_result.selected_indicators == second_result.selected_indicators

gates = []
def add_gate(name, passed, evidence):
    gates.append({"质量门槛": name, "通过": bool(passed), "证据": str(evidence)})

add_gate("数据库路径为项目主库", DATABASE_PATH.parent == PROJECT_ROOT / "data", str(DATABASE_PATH))
add_gate("三类财务指标均有通过项", all(selected[key] for key in ["income", "balance", "cashflow"]), selected)
add_gate("分红指标有通过项", bool(selected["dividend"]), selected["dividend"])
add_gate("第一次正式请求无错误", not first_result.errors, first_result.errors or "errors={}")
add_gate("财务事实行数符合预期", len(statement_facts) == expected_statement_rows and expected_statement_rows > 0, f"actual={len(statement_facts)}, expected={expected_statement_rows}")
add_gate("分红事实行数符合预期", len(dividend_facts) == expected_dividend_rows and expected_dividend_rows > 0, f"actual={len(dividend_facts)}, expected={expected_dividend_rows}")
add_gate("财务覆盖全部样本证券", statement_facts["symbol"].nunique() == len(SYMBOLS) if not statement_facts.empty else False, f"actual={statement_facts['symbol'].nunique() if not statement_facts.empty else 0}, expected={len(SYMBOLS)}")
add_gate("财务覆盖全部报告期", statement_facts["report_date"].nunique() == len(REPORT_DATES) if not statement_facts.empty else False, f"actual={statement_facts['report_date'].nunique() if not statement_facts.empty else 0}, expected={len(REPORT_DATES)}")
add_gate("财务覆盖三张报表", set(statement_facts["statement_type"]) == {"income", "balance", "cashflow"} if not statement_facts.empty else False, sorted(statement_facts["statement_type"].unique()) if not statement_facts.empty else [])
add_gate("财务事实存在非空值", int(statement_facts["has_value"].sum()) > 0 if not statement_facts.empty else False, f"non_null={int(statement_facts['has_value'].sum()) if not statement_facts.empty else 0}")
add_gate("分红事实存在非空值", int(dividend_facts["has_value"].sum()) > 0 if not dividend_facts.empty else False, f"non_null={int(dividend_facts['has_value'].sum()) if not dividend_facts.empty else 0}")
add_gate("财务事实无重复主键", statement_duplicates == 0, f"duplicates={statement_duplicates}")
add_gate("分红事实无重复主键", dividend_duplicates == 0, f"duplicates={dividend_duplicates}")
add_gate("来源均为Choice", set(statement_facts.get("source", [])) <= {"choice"} and set(dividend_facts.get("source", [])) <= {"choice"}, "source=choice")
add_gate("单位保持厂商原始尺度", set(statement_facts.get("unit", [])) <= {"vendor_raw"} and set(dividend_facts.get("unit", [])) <= {"vendor_raw"}, "unit=vendor_raw")
add_gate("数值字段无NaN或无穷值", finite_numeric, f"numeric_checked={len(numeric_values)}")
add_gate("第二次正式请求无错误", second_result is not None and not second_result.errors, second_result.errors if second_result else "未执行")
add_gate("第二次运行核心表不增长", second_core_unchanged, idempotency.to_dict("records"))
add_gate("两次指标探测结论一致", second_selection_same, "selected indicators一致" if second_selection_same else "不一致或未执行")
add_gate("SQLite完整性检查通过", scalar("PRAGMA quick_check") == "ok", scalar("PRAGMA quick_check"))

quality_gates = pd.DataFrame(gates)
passed_count = int(quality_gates["通过"].sum())
failed_count = int((~quality_gates["通过"]).sum())
print(f"质量门槛：{passed_count}项通过，{failed_count}项失败")
display(quality_gates)


## 导出Excel和JSON证据

In [ ]:
data_map = pd.DataFrame([
    {"dataset": "financial_statement_fact", "grain": "source+symbol+statement_type+report_date+indicator", "purpose": "利润表/资产负债表/现金流量表原始尺度事实", "unit_policy": "vendor_raw，确认后再标准化", "source": "choice"},
    {"dataset": "dividend_fact", "grain": "source+symbol+report_date+indicator", "purpose": "分红方案与日期类原始事实", "unit_policy": "vendor_raw，空值不填0", "source": "choice"},
    {"dataset": "financial_ingestion_run", "grain": "run_id", "purpose": "指标探测、下载和错误审计", "unit_policy": "不适用", "source": "choice"},
])

overview = pd.DataFrame([
    {"项目": "运行时间", "值": datetime.now(timezone.utc).isoformat()},
    {"项目": "数据库", "值": str(DATABASE_PATH)},
    {"项目": "备份", "值": str(backup_path) if backup_path else "未创建"},
    {"项目": "证券", "值": ",".join(SYMBOLS)},
    {"项目": "报告期", "值": ",".join(REPORT_DATES)},
    {"项目": "选中指标数", "值": sum(len(value) for value in selected.values())},
    {"项目": "被拒指标数", "值": sum(len(value) for value in first_result.rejected_indicators.values())},
    {"项目": "财务事实", "值": len(statement_facts)},
    {"项目": "分红事实", "值": len(dividend_facts)},
    {"项目": "质量门槛", "值": f"{passed_count}通过/{failed_count}失败"},
])

excel_path = OUTPUT_DIR / f"Choice财务报表分红小样本验收_{run_timestamp}.xlsx"
json_path = OUTPUT_DIR / f"Choice财务报表分红小样本验收_{run_timestamp}.json"

sheets = {
    "验收总览": overview,
    "质量门槛": quality_gates,
    "两次运行计数": idempotency,
    "选中指标": selected_indicators,
    "被拒指标": rejected_indicators,
    "请求错误": errors_frame,
    "财务事实": statement_facts.drop(columns=["has_value"], errors="ignore"),
    "分红事实": dividend_facts.drop(columns=["has_value"], errors="ignore"),
    "财务完整性": statement_completeness,
    "分红完整性": dividend_completeness,
    "运行日志": ingestion_runs,
    "数据地图": data_map,
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=sheet_name[:31], index=False)

from openpyxl import load_workbook
workbook = load_workbook(excel_path)
for worksheet in workbook.worksheets:
    worksheet.freeze_panes = "A2"
    worksheet.auto_filter.ref = worksheet.dimensions
    for column_cells in worksheet.columns:
        values = [str(cell.value or "") for cell in list(column_cells)[:200]]
        width = min(max(max((len(value) for value in values), default=8) + 2, 10), 50)
        worksheet.column_dimensions[column_cells[0].column_letter].width = width
workbook.save(excel_path)

json_payload = {
    "metadata": safe_config,
    "first_ingestion": first_summary,
    "second_ingestion": second_summary,
    "quality_summary": {"passed": passed_count, "failed": failed_count},
    "quality_gates": quality_gates.to_dict("records"),
    "idempotency_counts": idempotency.to_dict("records"),
    "selected_indicators": selected_indicators.to_dict("records"),
    "rejected_indicators": rejected_indicators.to_dict("records"),
    "errors": errors_frame.to_dict("records"),
    "statement_completeness": statement_completeness.to_dict("records"),
    "dividend_completeness": dividend_completeness.to_dict("records"),
    "data_map": data_map.to_dict("records"),
}
json_path.write_text(json.dumps(json_payload, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print("Excel：", excel_path)
print("JSON：", json_path)


## 结果解释

- 全部通过：说明小样本的指标权限、结构、落库和幂等性成立；下一步仍需在Choice终端核对单位和字段中文含义。
- “被拒指标”不为空但质量门槛全通过：正常，表示候选池里有不适用于当前版本或账号的指标。
- 某类指标全部未通过：打开Choice命令生成器，为同一证券和报告期生成`css`命令，替换对应`.env`指标列表后重跑。
- 分红非空值为0：不要填0，需要判断是报告期无方案，还是指标/`PayYear`参数不匹配。


In [ ]:
print(f"最终结论：{passed_count}项通过，{failed_count}项失败")
if failed_count == 0:
    print("✅ 09号Choice财务报表与分红小样本验收通过。")
else:
    print("⚠️ 已保存失败证据，请查看“质量门槛”“被拒指标”和“请求错误”。")

if STRICT_MODE and failed_count:
    failed_names = quality_gates.loc[~quality_gates["通过"], "质量门槛"].tolist()
    raise RuntimeError(f"严格模式：以下质量门槛未通过：{failed_names}")
